# Step 2a · Wind, rain and temperature for each county and week

**Purpose**: Step 1b found that the counties where visits collapsed were almost all counties left without power for close to a whole week, and these were also the counties hit hardest by the wind. To separate "the effect of the outage" from "the effect of the storm itself", we need to know how strong the wind was, how much rain fell and how cold it was in each county and week. This step only gathers those data; the models are in Step 2b.

**Data** (all on Earth Engine)

| Data | What is taken | Resolution |
| --- | --- | --- |
| RTMA (the U.S. National Weather Service's hourly analysis) | Highest gust of the week: mean and maximum within the county (m/s) | 2.5 km, hourly |
| PRISM (Oregon State University's daily grids) | Weekly rainfall total, wettest day (mm), lowest temperature (°C), frozen precipitation (mm) | About 4 km, daily |

**Definitions**

- **Weeks** are as in Step 1b: local Monday to Sunday, 12 weeks per event (weeks −5 to 6)
- **Gust**: at each grid point, the highest gust among the week's 168 hours is taken first; then its mean over the county (`gust_mean_ms`, the county as a whole) and its maximum (`gust_max_ms`, the worst place in the county)
- **A PRISM day**: the value labeled day D covers the 24 hours ending at 12:00 UTC on D (7 a.m. Eastern), so it corresponds mostly to local day D−1. Days are assigned to weeks accordingly
- **Frozen precipitation** (`cold_ppt_mm`): precipitation on days whose maximum temperature was 2 °C or less, a rough stand-in for freezing rain and snow in the ice storm and the winter storm

**Structure**

| Part | Content | Approx. time |
| --- | --- | --- |
| A | Read the events and counties from Step 1b | 1 minute |
| B | Connect to Earth Engine | 1 minute |
| C | RTMA gusts (8 events) | 15–40 minutes |
| D | PRISM rainfall and temperature (8 events) | 5–15 minutes |
| E | Merge, checks, one figure, checklist | 1 minute |

Each event is saved to Drive as soon as it is done, so a rerun after an interruption skips the events already finished.

**Running the notebook**: set `PROJECT_ID` in cell 2 to an Earth Engine Cloud project ID.

In [ ]:
# 1. Environment, folders, chart style
import os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:                     # in local Jupyter: use the current folder
    BASE = "."
OUT = f"{BASE}/outage_mobility"
CACHE = f"{OUT}/step2a_cache"           # intermediate results, one file per event and source
os.makedirs(CACHE, exist_ok=True)
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run Step 1b first.")

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank()) if ok.sum() >= 5 else np.nan

def cached(name, compute):
    # read the result from the cache folder if it is there, otherwise compute and save it
    path = f"{CACHE}/{name}.csv"
    if not os.path.exists(path):
        compute().to_csv(path, index=False)
    return pd.read_csv(path, dtype={"fips": str})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_POINT = "#2a78d6"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
PROJECT_ID = "ee-yourname"       # ← an Earth Engine Cloud project ID

WEEK_FROM, WEEK_TO = -5, 6       # as in Step 1b
EE_BATCH  = 100                  # counties per Earth Engine request; a failed request is split in half automatically
MIN_BATCH = 5                    # if a request this small still fails, stop with an error
RTMA_SCALE, PRISM_SCALE = 2500, 4000     # resolution used for the computation (m), the native resolution of each dataset
FINE_SCALE = 500                 # very small counties (such as Virginia's independent cities) may get no value; they are retried at this finer resolution
COLD_TMAX = 2.0                  # precipitation on days with a maximum temperature at or below this counts as frozen

## A. Read the events and counties from Step 1b

Events, time zones, week 0 and each event's counties come straight from the files Step 1b saved, so that the weeks of the two steps line up exactly.

In [ ]:
# 3. Events, weeks, counties
STATE_FIPS = {"AL": "01", "AR": "05", "CT": "09", "DE": "10", "GA": "13", "IL": "17", "IN": "18", "IA": "19",
              "KS": "20", "LA": "22", "MD": "24", "MA": "25", "MS": "28", "MO": "29", "NE": "31", "NJ": "34",
              "NY": "36", "NC": "37", "OK": "40", "PA": "42", "RI": "44", "SC": "45", "TX": "48", "VA": "51", "WI": "55"}
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
EVENTS["states"] = [[STATE_FIPS[a] for a in s.split()] for s in EVENTS["states"]]
for c in ["t0", "day0", "week0"]:
    EVENTS[c] = pd.to_datetime(EVENTS[c])
ALL_STATES = sorted({s for st in EVENTS["states"] for s in st})
labels = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})

def event_weeks(key):
    # the event's 12 Mondays (local dates)
    w0 = EVENTS.loc[key, "week0"]
    return [w0 + pd.Timedelta(weeks=r) for r in range(WEEK_FROM, WEEK_TO + 1)]

def event_fips(key):
    return sorted(labels.loc[labels["event"] == key, "fips"])

def utc_text(local_midnight, tz):
    # a local midnight → UTC time string (daylight saving handled)
    return local_midnight.tz_localize(tz).tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%S")

N_WEEKS = WEEK_TO - WEEK_FROM + 1
EVENTS[["name", "tz", "week0"]].assign(counties=[len(event_fips(k)) for k in EVENTS.index])

## B. Connect to Earth Engine

County boundaries come from `TIGER/2018/Counties` on Earth Engine, which uses the county codes from before 2022, the same codes EAGLE-I and GeoDS use in 2020–2021.

`county_reduce` takes the mean of a multi-band image within each county (or the mean and the maximum together), 100 counties per request. A failed request (a computation timeout, say) is split in half and tried again; a batch of 5 counties that still fails is retried after a pause, and only three failures in a row stop the notebook. If a county gets no value (too small for any grid point to fall inside), it is fetched again at 500 m resolution.

In [ ]:
# 4. [EE] Sign in to Earth Engine; the function that takes values by county
import ee
ee.Authenticate()                         # follow the link to authorize
ee.Initialize(project=PROJECT_ID)
counties_fc = ee.FeatureCollection("TIGER/2018/Counties").filter(ee.Filter.inList("STATEFP", ALL_STATES))

def ee_rows(fc, props, tries=3):
    # fetch a few properties of a feature collection (Earth Engine drops features where any of them is empty)
    for attempt in range(tries):
        try:
            return fc.reduceColumns(ee.Reducer.toList(len(props)), props).get("list").getInfo()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"  request failed, retrying in {15 * (attempt + 1)} s:", str(err)[:150])
            time.sleep(15 * (attempt + 1))

def out_names(bands, how):
    # the names Earth Engine gives the results: the band names for the mean alone; "band_mean" and "band_max" for mean and maximum together
    return list(bands) if how == "mean" else [f"{b}_mean" for b in bands] + [f"{b}_max" for b in bands]

def county_reduce(img, bands, fips, scale, how="mean"):
    # the mean (how="mean"), or the mean and the maximum (how="mean_max"), of each band within each county
    reducer = ee.Reducer.mean() if how == "mean" else ee.Reducer.mean().combine(ee.Reducer.max(), sharedInputs=True)
    names, rows = out_names(bands, how), []
    queue = [fips[i:i + EE_BATCH] for i in range(0, len(fips), EE_BATCH)]
    while queue:
        part = queue.pop(0)
        try:
            stats = img.select(bands).reduceRegions(collection=counties_fc.filter(ee.Filter.inList("GEOID", part)),
                                                    reducer=reducer, scale=scale, tileScale=4)
            rows += ee_rows(stats, ["GEOID"] + names, tries=1 if len(part) > MIN_BATCH else 3)   # a large batch is split straight away rather than retried
        except Exception as err:
            if len(part) <= MIN_BATCH:
                raise
            half = len(part) // 2
            queue = [part[:half], part[half:]] + queue
            print(f"  a request for {len(part)} counties failed; splitting it in half:", str(err)[:120])
    out = pd.DataFrame(rows, columns=["fips"] + names).drop_duplicates("fips").set_index("fips")
    missing = [f for f in fips if f not in out.index]
    if missing and scale > FINE_SCALE:
        print(f"  {len(missing)} counties got no value; retrying at {FINE_SCALE} m:", missing[:8])
        extra = county_reduce(img, bands, missing, FINE_SCALE, how).set_index("fips")
        out = pd.concat([out, extra.dropna(how="all")])
    return out.reindex(fips).rename_axis("fips").reset_index()
print("Earth Engine connected")

## C. RTMA gusts

For each event, one 12-band image is built; band i holds the highest gust of week i at each grid point. Each week starts at local Monday midnight, converted to UTC before the hourly RTMA images are filtered. The occasional hour without a gust band is skipped.

Results: `gust_mean_ms` (mean within the county) and `gust_max_ms` (maximum within the county), in m/s. For reference, 17.5 m/s is about the lower bound of tropical-storm sustained winds and 33 m/s about hurricane force.

In [ ]:
# 5. [EE] RTMA: highest gust per event, county and week
rtma = ee.ImageCollection("NOAA/NWS/RTMA")
GUST_BANDS = [f"g{i:02d}" for i in range(N_WEEKS)]

def rtma_event(key):
    tz, weeks = EVENTS.loc[key, "tz"], event_weeks(key)
    bands = []
    for b, w in zip(GUST_BANDS, weeks):
        hours = (rtma.filterDate(utc_text(w, tz), utc_text(w + pd.Timedelta(days=7), tz))
                     .filter(ee.Filter.listContains("system:band_names", "GUST")))
        bands.append(hours.select("GUST").max().rename(b))
    return county_reduce(ee.Image.cat(bands), GUST_BANDS, event_fips(key), RTMA_SCALE, "mean_max").assign(event=key)

gust_parts = []
for key in EVENTS.index:
    t = time.time()
    gust_parts.append(cached(f"rtma_{key}", lambda: rtma_event(key)))
    have = gust_parts[-1][[f"{b}_mean" for b in GUST_BANDS]].notna().all(axis=1).mean()
    print(f"{key}: {have:.1%} of counties have all 12 weeks ({time.time() - t:.0f} s)")

## D. PRISM rainfall and temperature

First all daily PRISM images in each event's window are listed and grouped by the date in the image ID (PRISM's date label). Occasionally a date has two versions of an image; they are averaged. Each local day is then matched to the PRISM label of the following day, and the days are summed by week into four numbers:

| Column | Meaning |
| --- | --- |
| `rain_mm` | Total precipitation in the week |
| `rain_max1d_mm` | The wettest day of the week |
| `tmin_c` | The lowest temperature of the week |
| `cold_ppt_mm` | Total precipitation on the week's days with a maximum temperature ≤ 2 °C |

The IDs and start times of a few images are printed first, to show how the dates were matched.

In [ ]:
# 6. [EE] PRISM: rainfall and temperature per event, county and week
prism = ee.ImageCollection("OREGONSTATE/PRISM/ANd")
WEEK_BANDS = {kind: [f"{kind}{i:02d}" for i in range(N_WEEKS)] for kind in ["r", "x", "t", "c"]}
ALL_PRISM_BANDS = [b for kind in "rxtc" for b in WEEK_BANDS[kind]]

def prism_index(key):
    # the ID and date label of every PRISM image in the window
    w = event_weeks(key)
    coll = prism.filterDate((w[0] - pd.Timedelta(days=2)).strftime("%Y-%m-%d"), (w[-1] + pd.Timedelta(days=10)).strftime("%Y-%m-%d"))
    ids, starts = coll.aggregate_array("system:index").getInfo(), coll.aggregate_array("system:time_start").getInfo()
    tab = pd.DataFrame({"id": ids, "start": pd.to_datetime(starts, unit="ms")})
    found = tab["id"].astype(str).str.extract(r"(\d{8})")[0]
    tab["label"] = pd.to_datetime(found, format="%Y%m%d", errors="coerce")
    if tab["label"].isna().any():
        print("  some image IDs hold no date; the date of the start time is used as the label instead")
        tab["label"] = tab["label"].fillna(tab["start"].dt.normalize())
    return tab

def prism_event(key):
    tab = prism_index(key)
    print(f"{key}: first 3 images:", tab.head(3).to_dict("records"))
    day_img, missing = {}, []
    def day(label):
        # the image for one PRISM date label (versions averaged), plus the frozen-precipitation band
        ids = tab.loc[tab["label"] == label, "id"].tolist()
        if not ids:
            missing.append(label)
            return None
        if label not in day_img:
            im = prism.filter(ee.Filter.inList("system:index", ids)).mean()
            day_img[label] = im.select(["ppt", "tmin"]).addBands(im.select("ppt").multiply(im.select("tmax").lte(COLD_TMAX)).rename("cold"))
        return day_img[label]
    bands, n_days = [], {}
    for i, w in enumerate(event_weeks(key)):
        days = [d for d in (day(w + pd.Timedelta(days=j + 1)) for j in range(7)) if d is not None]   # local day j ↔ PRISM label j+1
        n_days[i] = len(days)
        week = ee.ImageCollection.fromImages(days)
        bands += [week.select("ppt").sum().rename(WEEK_BANDS["r"][i]), week.select("ppt").max().rename(WEEK_BANDS["x"][i]),
                  week.select("tmin").min().rename(WEEK_BANDS["t"][i]), week.select("cold").sum().rename(WEEK_BANDS["c"][i])]
    if missing:
        print(f"  {len(missing)} PRISM days have no image:", [d.strftime("%Y-%m-%d") for d in missing][:10])
    img = ee.Image.cat(bands)
    out = county_reduce(img, ALL_PRISM_BANDS, event_fips(key), PRISM_SCALE, "mean").assign(event=key)
    for i, n in n_days.items():
        out[f"n{i:02d}"] = n                                                     # how many days the week actually used
    return out

prism_parts = []
for key in EVENTS.index:
    t = time.time()
    prism_parts.append(cached(f"prism_{key}", lambda: prism_event(key)))
    have = prism_parts[-1][WEEK_BANDS["r"]].notna().all(axis=1).mean()
    print(f"{key}: {have:.1%} of counties have all 12 weeks ({time.time() - t:.0f} s)")

## E. Merge, checks, checklist

The two results are turned into one long "event × county × week" table (`step2a_hazard_weekly.csv`) and set beside Step 1b's outage figures to look at two things:

- Whether the numbers for a few hard-hit counties are plausible (Calcasieu's gust in Laura's week 0 should be near or above hurricane force, and Texas's lowest temperature during Uri far below 0 °C)
- Whether storm strength goes with outage duration within each event. The figure uses the highest gust for the windstorms, frozen precipitation for the ice storm and the lowest temperature for Uri

In [ ]:
# 7. Merge into one long table
def to_long(parts, columns):
    # wide table (one column per week) → long table (one row per week)
    rows = []
    for w in parts:
        for i in range(N_WEEKS):
            one = w[["event", "fips"]].copy()
            one["rel_week"] = i + WEEK_FROM
            for new, old in columns.items():
                one[new] = w[old.format(i=i)] if old.format(i=i) in w else np.nan
            rows.append(one)
    return pd.concat(rows, ignore_index=True)

gust = to_long(gust_parts, {"gust_mean_ms": "g{i:02d}_mean", "gust_max_ms": "g{i:02d}_max"})
rain = to_long(prism_parts, {"rain_mm": "r{i:02d}", "rain_max1d_mm": "x{i:02d}", "tmin_c": "t{i:02d}",
                             "cold_ppt_mm": "c{i:02d}", "prism_days": "n{i:02d}"})
hz = gust.merge(rain, on=["event", "fips", "rel_week"], how="outer")
hz["week"] = hz["event"].map(EVENTS["week0"]) + pd.to_timedelta(hz["rel_week"] * 7, unit="D")
hz = hz.sort_values(["event", "fips", "rel_week"]).reset_index(drop=True)
hz.to_csv(f"{OUT}/step2a_hazard_weekly.csv", index=False)
print(f"Hazard table: {len(hz):,} rows (event × county × week)")
tidy(hz[hz["rel_week"].isin([0, 1])].groupby("event")[["gust_mean_ms", "gust_max_ms", "rain_mm", "tmin_c", "cold_ppt_mm"]]
      .agg(["median", "max"]).reindex(EVENTS.index))

In [ ]:
# 8. Numbers for a few hard-hit counties (weeks −1 to 2)
SPOTS = [("laura2020", "22019"), ("delta2020", "22023"), ("derecho2020", "19113"), ("isaias2020", "09001"),
         ("okice2020", "40109"), ("zeta2020", "22087"), ("uri2021", "48201"), ("ida2021", "22109")]
spot = hz[hz["rel_week"].between(-1, 2)].set_index(["event", "fips"]).loc[[s for s in SPOTS if s in set(zip(hz["event"], hz["fips"]))]]
names = labels.set_index(["event", "fips"])["name"]
spot = spot.reset_index().assign(name=lambda d: [names.get((e, f), f) for e, f in zip(d["event"], d["fips"])])
tidy(spot.pivot_table(index=["event", "name"], columns="rel_week", values=["gust_mean_ms", "rain_mm", "tmin_c"]), 1)

In [ ]:
# 9. Figure: storm strength and outage duration (weeks 0 and 1)
panel = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", usecols=["event", "fips", "rel_week", "out_week_hours"], dtype={"fips": str})
d01 = (panel[panel["rel_week"].isin([0, 1])].groupby(["event", "fips"])["out_week_hours"].sum().rename("out_hours").reset_index()
       .merge(hz[hz["rel_week"].isin([0, 1])].groupby(["event", "fips"]).agg(gust=("gust_mean_ms", "max"), cold=("cold_ppt_mm", "sum"),
                                                                            tmin=("tmin_c", "min")).reset_index(), on=["event", "fips"]))
XVAR = {k: ("gust", "Highest gust, county mean (m/s)") for k in EVENTS.index}
XVAR["okice2020"] = ("cold", "Frozen precipitation (mm)")
XVAR["uri2021"] = ("tmin", "Lowest temperature (°C)")
rho = {}
fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
for ax, key in zip(axes.flat, EVENTS.index):
    s = d01[d01["event"] == key]
    col, xlabel = XVAR[key]
    x = -s[col] if col == "tmin" else s[col]                          # colder is more severe, so the sign is flipped for the correlation
    rho[key] = spearman(x, s["out_hours"])
    ax.scatter(s[col], s["out_hours"], s=9, color=C_POINT, alpha=0.5, edgecolor="none")
    ax.set_yscale("symlog", linthresh=1, linscale=0.5)                # linear near 0, logarithmic above
    ax.set_yticks([0, 1, 3, 10, 30, 100, 300])
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.set_title(f"{EVENTS.loc[key, 'name']} {pd.Timestamp(EVENTS.loc[key, 't0']).year}")
    ax.text(0.03, 0.95, f"Spearman {rho[key]:.2f}", transform=ax.transAxes, va="top", color=MUTED, fontsize=9,
            bbox=dict(facecolor=SURFACE, edgecolor="none", alpha=0.85, pad=1.5))
    ax.set_xlabel(xlabel)
    if col == "tmin":
        ax.invert_xaxis()
for ax in axes[:, 0]:
    ax.set_ylabel("Hours without power per customer,\nweeks 0 and 1 (log scale above 1)")
fig.suptitle("Storm strength and outage duration by county, weeks 0 and 1", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step2a_fig_hazard_vs_outage.png"); plt.show()

**Reading the figure**: each dot is a county. For the windstorms, stronger gusts should go with longer outages (in the companion outage-duration project, gusts alone explained most of the ranking of outage durations). The ice storm's horizontal axis is frozen precipitation, and Uri's is the lowest temperature (colder to the right). Uri's outages came mainly from ERCOT's rotating outages and may have little to do with how cold a given place was, which is one reason it suits only the transfer test.

In [ ]:
# 10. Step 2a checklist
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

gust_cover = float(hz["gust_mean_ms"].notna().mean())
rain_cover = float(hz["rain_mm"].notna().mean())
short_weeks = int((hz.drop_duplicates(["event", "rel_week"])["prism_days"] < 7).sum())
WIND = [k for k in EVENTS.index if EVENTS.loc[k, "kind"] in ("hurricane", "derecho")]
rho_min = float(np.nanmin([rho[k] for k in WIND]))
uri_tx = hz[(hz["event"] == "uri2021") & (hz["rel_week"] == 0) & hz["fips"].str.startswith("48")]["tmin_c"].median()
ice = d01[(d01["event"] == "okice2020") & d01["fips"].str.startswith("40") & (d01["out_hours"] >= 24)]["cold"].median()
checks = [
    ("Share of county-weeks with gust data", round(gust_cover, 4), "≥ 0.99", gust_cover >= 0.99),
    ("Share of county-weeks with rainfall data", round(rain_cover, 4), "≥ 0.99", rain_cover >= 0.99),
    ("Event-weeks with fewer than 7 PRISM days", short_weeks, "= 0", short_weeks == 0),
    ("Spearman between gust and outage duration, lowest windstorm", round(rho_min, 2), "≥ 0.3", rho_min >= 0.3),
    ("Uri, week 0: lowest temperature in Texas counties (median, °C)", round(float(uri_tx), 1), "≤ −10", bool(uri_tx <= -10)),
    ("Ice storm: frozen precipitation in Oklahoma counties with 24+ h outages (median, mm)",
     None if pd.isna(ice) else round(float(ice), 1), "≥ 10", None if pd.isna(ice) else bool(ice >= 10)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step2a_checklist.csv", index=False)
check_tab